In [2]:
# ============================================================
# Cellule 1 — Imports (numérotés et commentés)
# ============================================================

# [1] Core Python & Data ------------------------------------------------------
import numpy as np                      # [1.1] Calcul numérique (vecteurs, quantiles, etc.)
import pandas as pd                     # [1.2] DataFrames / séries temporelles
from dateutil.relativedelta import relativedelta  # [1.3] Décalages mensuels propres (ex: +12 mois)

import json                             # [1.4] Config / meta (sauvegarde params)
import warnings                         # [1.5] Gérer/masquer warnings (statsmodels, etc.)
import logging                          # [1.6] Logs (CV updates, suivi p*)
import io                               # [1.7] Buffer texte (capturer stdout/stderr)
from contextlib import redirect_stdout, redirect_stderr  # [1.8] Redirection des sorties


# [2] Custom Utilities (Feast + Pipeline AR(p)) -------------------------------
from utils import (
    load_wide_from_feast,               # [2.1] Charger les séries depuis Feast (feature store)
    run_pseudo_oos_ar_p_no_bagging       # [2.2] Backtest pseudo-OOS AR(p) (sans bagging)
)


# [3] Modeling ----------------------------------------------------------------
from statsmodels.tsa.ar_model import AutoReg    # [3.1] Modèle économétrique AR(p)
from mlforecast.utils import PredictionIntervals # [3.2] (Optionnel) utilitaires d'intervalles


# [4] Metrics -----------------------------------------------------------------
from sklearn.metrics import mean_absolute_error # [4.1] MAE = erreur moyenne absolue


# [5] MLOps / Tracking --------------------------------------------------------
import mlflow                           # [5.1] Tracking des expériences (params/metrics/artefacts)

In [3]:
# ============================================================
# Cellule 2 — Charger uniquement UNRATE (Feast) + ts_ar propre
# ============================================================
series_ids = ["UNRATE"]
START = "1960-01-01"
END   = "2025-08-01"

df = load_wide_from_feast(
    "stationary_value:value",
    series_ids,
    start=START,
    end=END
)

ts_ar = (
    df.reset_index()
      .rename(columns={"date": "ds", "UNRATE": "y"})
)

ts_ar["unique_id"] = "UNRATE"
ts_ar["ds"] = pd.to_datetime(ts_ar["ds"], errors="coerce")

# ✅ Fix timezone (si UTC)
if getattr(ts_ar["ds"].dt, "tz", None) is not None:
    ts_ar["ds"] = ts_ar["ds"].dt.tz_convert(None)

# ✅ Normaliser MS
ts_ar["ds"] = ts_ar["ds"].dt.to_period("M").dt.to_timestamp(how="start").dt.normalize()

ts_ar = (
    ts_ar[["unique_id", "ds", "y"]]
    .dropna(subset=["ds", "y"])
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

print("ts_ar shape:", ts_ar.shape)
print("Range:", ts_ar["ds"].min().date(), "→", ts_ar["ds"].max().date())
ts_ar.head()

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
ts_ar shape: (788, 3)
Range: 1960-01-01 → 2025-08-01


series_id,unique_id,ds,y
0,UNRATE,1960-01-01,-0.8
1,UNRATE,1960-02-01,-1.1
2,UNRATE,1960-03-01,-0.2
3,UNRATE,1960-04-01,0.0
4,UNRATE,1960-05-01,0.0


In [4]:
def ts_to_y(ts_ar: pd.DataFrame, uid="UNRATE") -> pd.Series:
    # filtre la série
    df = ts_ar.loc[ts_ar["unique_id"].eq(uid), ["ds", "y"]].copy()
    df["ds"] = pd.to_datetime(df["ds"])

    # force début de mois (MS) et trie
    df["ds"] = df["ds"].dt.to_period("M").dt.to_timestamp(how="start")
    df = df.sort_values("ds")

    # Series avec index DatetimeIndex
    y = df.set_index("ds")["y"].astype(float)

    # (optionnel) s'assurer que la fréquence est mensuelle MS
    y = y.asfreq("MS")
    return y

y = ts_to_y(ts_ar, uid="UNRATE")
print(y.head())

ds
1960-01-01   -0.8
1960-02-01   -1.1
1960-03-01   -0.2
1960-04-01    0.0
1960-05-01    0.0
Freq: MS, Name: y, dtype: float64


# AR12

In [5]:
df_oos_ar, meta = run_pseudo_oos_ar_p_no_bagging(
    y,
    h=12,
    min_train_n=36,
    trend="c",
    p_grid=range(1, 13),
    cv_update_every_months=36,
    cv_anchor=pd.Timestamp("1983-01-01"),
    use_conformal=True,
    alpha=0.05,
    step_size=12,
    pi_windows=3,
)

[CV] 1983-01-01 → p* = 5
[CV] 1986-01-01 → p* = 4
[CV] 1989-01-01 → p* = 4
[CV] 1992-01-01 → p* = 4
[CV] 1995-01-01 → p* = 4
[CV] 1998-01-01 → p* = 4
[CV] 2001-01-01 → p* = 4
[CV] 2004-01-01 → p* = 4
[CV] 2007-01-01 → p* = 4
[CV] 2010-01-01 → p* = 4
[CV] 2013-01-01 → p* = 4
[CV] 2016-01-01 → p* = 4
[CV] 2019-01-01 → p* = 4
[CV] 2022-01-01 → p* = 4


In [6]:
df_oos_ar = df_oos_ar.drop("y_hat_base", axis=1)

In [7]:
df_oos_ar

,y_hat,y_true,p_selected,lo_95,hi_95
date,,,,,
1963-12-01,-0.080890,0.0,1,NaN,NaN
1964-01-01,0.141077,-0.1,1,NaN,NaN
1964-02-01,0.408114,-0.5,1,NaN,NaN
1964-03-01,0.242637,-0.3,1,NaN,NaN
1964-04-01,0.238955,-0.4,1,NaN,NaN
...,...,...,...,...,...
2025-04-01,0.086570,0.3,4,-0.495027,0.668166
2025-05-01,0.059958,0.2,4,-0.528360,0.648276
2025-06-01,0.085175,0.0,4,-1.116228,1.286579


# Transformer le Backtesting

In [9]:
# =========================
# Paramètres période évaluation
# =========================
EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")

# =========================
# bkt_score depuis df_oos_ar (index=date)
# =========================
bkt_score = df_oos_ar.copy()

# index -> colonne ds (comme tes autres pipelines)
bkt_score = bkt_score.reset_index().rename(columns={"date": "ds"})
bkt_score["ds"] = pd.to_datetime(bkt_score["ds"], errors="coerce")

# enlever timezone si besoin
if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):
    bkt_score["ds"] = bkt_score["ds"].dt.tz_convert(None)

bkt_score = bkt_score.dropna(subset=["ds"])

# filtre période
bkt_score = bkt_score[
    (bkt_score["ds"] >= EXP_START) &
    (bkt_score["ds"] <= EXP_END)
].reset_index(drop=True)

# =========================
# Partitions (mêmes bins/labels que toi)
# =========================
bins = pd.to_datetime([
    "1990-01-01",
    "2000-01-01",
    "2009-01-01",
    "2020-01-01",
    "2025-09-01"
])

labels = [
    "1990-1999",
    "2000-2008",
    "2009-2019",
    "2020-end"
]

bkt_score["partition"] = pd.cut(
    bkt_score["ds"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

bkt_score = bkt_score.dropna(subset=["partition"]).reset_index(drop=True)

print(bkt_score[["ds","partition","p_selected"]].head(10))
print(bkt_score["partition"].value_counts().sort_index())

          ds  partition  p_selected
0 1990-01-01  1990-1999           4
1 1990-02-01  1990-1999           4
2 1990-03-01  1990-1999           4
3 1990-04-01  1990-1999           4
4 1990-05-01  1990-1999           4
5 1990-06-01  1990-1999           4
6 1990-07-01  1990-1999           4
7 1990-08-01  1990-1999           4
8 1990-09-01  1990-1999           4
9 1990-10-01  1990-1999           4
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-end      68
Name: count, dtype: int64


C:\Users\Mita\AppData\Local\Temp\ipykernel_17120\2582519751.py:17: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):


In [18]:
bkt_score

,ds,y_hat,y_true,p_selected,lo_95,hi_95,partition
0,1990-01-01,0.038995,0.0,4,-0.896105,0.974096,1990-1999
1,1990-02-01,-0.158664,0.1,4,-0.963583,0.646255,1990-1999
2,1990-03-01,-0.338061,0.2,4,-1.205755,0.529634,1990-1999
3,1990-04-01,0.079964,0.2,4,-0.698902,0.858830,1990-1999
4,1990-05-01,-0.004353,0.2,4,-0.955797,0.947091,1990-1999
...,...,...,...,...,...,...,...
423,2025-04-01,0.086570,0.3,4,-0.495027,0.668166,2020-end
424,2025-05-01,0.059958,0.2,4,-0.528360,0.648276,2020-end
425,2025-06-01,0.085175,0.0,4,-1.116228,1.286579,2020-end
426,2025-07-01,0.129088,0.0,4,-0.858917,1.117093,2020-end


# Leaderbord

In [10]:
tmp = bkt_score.copy()

# -------------------------
# 0) Harmoniser colonnes (ARp)
# -------------------------
tmp["unique_id"] = tmp.get("unique_id", "UNRATE")
tmp["y"] = tmp["y_true"]
tmp["ARp"] = tmp["y_hat"]
tmp["ARp-lo-95"] = tmp["lo_95"]
tmp["ARp-hi-95"] = tmp["hi_95"]

models = ["ARp"]

# -------------------------
# 1) S'assurer que lower <= upper
# -------------------------
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"
    tmp[[lo, hi]] = np.sort(tmp[[lo, hi]].to_numpy(), axis=1)

# -------------------------
# 2) Wide -> Long + scoring
# -------------------------
rows = []
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"

    s = tmp[["unique_id", "ds", "y", "partition"]].copy()
    s["cutoff"] = pd.NaT
    s["model_label"] = m
    s["model_name"]  = m

    s["forecast"] = tmp[m]
    s["lower"]    = tmp[lo]
    s["upper"]    = tmp[hi]

    s["abs_err"] = (s["y"] - s["forecast"]).abs()

    has_pi = s["lower"].notna() & s["upper"].notna()
    s["covered"] = np.where(
        has_pi,
        ((s["y"] >= s["lower"]) & (s["y"] <= s["upper"])).astype(int),
        np.nan
    )
    s["int_width"] = np.where(
        has_pi,
        (s["upper"] - s["lower"]).abs(),
        np.nan
    )

    rows.append(s)

long_sc = pd.concat(rows, ignore_index=True)
long_sc["partition"] = long_sc["partition"].astype(str)
long_sc = long_sc.loc[:, ~long_sc.columns.duplicated()]

# -------------------------
# 3) Score par partition
# -------------------------
score_by_part = (
    long_sc
    .groupby(["unique_id", "model_label", "model_name", "partition"], observed=True)
    .agg(
        mae=("abs_err", "mean"),
        coverage=("covered", "mean"),
        width=("int_width", "mean"),
        n=("y", "size"),
    )
    .reset_index()
)

# -------------------------
# 4) Score ALL (toutes partitions)
# -------------------------
score_all = (
    long_sc
    .assign(partition="ALL")
    .groupby(["unique_id", "model_label", "model_name", "partition"], observed=True)
    .agg(
        mae=("abs_err", "mean"),
        coverage=("covered", "mean"),
        width=("int_width", "mean"),
        n=("y", "size"),
    )
    .reset_index()
)

score_df = (
    pd.concat([score_by_part, score_all], ignore_index=True)
    .sort_values(["partition", "mae"])
)

# -------------------------
# 5) Top 3 par partition + ALL
# -------------------------
leaderboard = (
    score_df.sort_values(
        by=["partition", "mae", "coverage", "width"],
        ascending=[True, True, False, True],
    )
    .groupby("partition", as_index=False)
    .head(3)
)

In [11]:
print("Score (partitions + ALL):")
print(score_df.sort_values(["partition", "mae"]).reset_index(drop=True))

Score (partitions + ALL):
  unique_id model_label model_name  partition       mae  coverage     width  \
0    UNRATE         ARp        ARp  1990-1999  0.494189  0.816667  1.763992   
1    UNRATE         ARp        ARp  2000-2008  0.514651  0.685185  1.569703   
2    UNRATE         ARp        ARp  2009-2019  0.763062  0.825758  3.086353   
3    UNRATE         ARp        ARp   2020-end  2.312401  0.661765  9.135620   
4    UNRATE         ARp        ARp        ALL  0.871151  0.761682  3.293990   

     n  
0  120  
1  108  
2  132  
3   68  
4  428  


# MLFLOW

In [19]:
import os
import joblib
import warnings
import logging
import io
import json
import pandas as pd
import mlflow
from contextlib import redirect_stdout, redirect_stderr
from statsmodels.tsa.ar_model import AutoReg

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Baseline")

df_log = score_df.copy()

AR_PARAMS = dict(
    horizon=12,
    min_train_n=36,
    trend="c",
    p_grid=list(range(1, 13)),
    cv_update_every_months=36,
    cv_anchor="1983-01-01",
    use_conformal=True,
    alpha=0.05,
    step_size=12,
    pi_windows=3,
    use_bagging=False,
)

METRIC_SEQUENCE = "mae>coverage>width"

# ------------------------------------------------------------
# Résumé partitions
# ------------------------------------------------------------
partition_summary_df = (
    bkt_score
    .groupby("partition", observed=True)
    .agg(
        n_obs=("ds", "size"),
        ds_start=("ds", "min"),
        ds_end=("ds", "max"),
    )
    .reset_index()
)

all_row = pd.DataFrame([{
    "partition": "ALL",
    "n_obs": int(len(bkt_score)),
    "ds_start": bkt_score["ds"].min(),
    "ds_end": bkt_score["ds"].max(),
}])

partition_summary_df = pd.concat([partition_summary_df, all_row], ignore_index=True)

dataset_obj = mlflow.data.from_pandas(
    partition_summary_df,
    source="pandas",
    name="feast:stationary_value:value | UNRATE partition summary"
)

# ------------------------------------------------------------
# Helper refit
# ------------------------------------------------------------
def fit_final_ar_for_partition(ts_df, ds_start, ds_end, p=12, trend="c"):
    if isinstance(ts_df, pd.DataFrame):
        y = ts_df.loc[(ts_df["ds"] >= ds_start) & (ts_df["ds"] <= ds_end), "y"].astype(float)
        y.index = pd.to_datetime(ts_df.loc[(ts_df["ds"] >= ds_start) & (ts_df["ds"] <= ds_end), "ds"])
        y = y.sort_index()
    else:
        y = ts_df.loc[ds_start:ds_end].astype(float).sort_index()

    if len(y) < (p + 5):
        raise ValueError(f"Pas assez de données pour fitter AR({p})")

    return AutoReg(y, lags=int(p), old_names=False, trend=trend).fit()

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
_sink = io.StringIO()

with redirect_stdout(_sink), redirect_stderr(_sink):

    for _, row in df_log.iterrows():

        partition = row["partition"]
        run_name = f"AR_{partition}"

        with mlflow.start_run(run_name=run_name):

            mlflow.log_input(dataset_obj, context="evaluation")

            # Partition summary
            csv_path = "partition_summary.csv"
            partition_summary_df.to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path)

            # Params
            mlflow.log_param("partition", partition)
            mlflow.log_param("metric_sequence", METRIC_SEQUENCE)
            for k, v in AR_PARAMS.items():
                mlflow.log_param(k, json.dumps(v) if isinstance(v, (list, dict)) else v)

            # Metrics
            mlflow.log_metric("mae", float(row["mae"]))
            if pd.notna(row.get("coverage", None)):
                mlflow.log_metric("coverage", float(row["coverage"]))
            if pd.notna(row.get("width", None)):
                mlflow.log_metric("width", float(row["width"]))

            # ------------------------------------------------
            # 🔥 Log bkt_score utile pour DM  (FIX: y_true, lo_95, hi_95)
            # ------------------------------------------------
            os.makedirs("artifacts_tmp", exist_ok=True)

            if "bkt_score" in globals():

                # 1) Par partition
                df_part = bkt_score[bkt_score["partition"].astype(str) == str(partition)].copy()
                if len(df_part) > 0:
                    path_part = f"artifacts_tmp/bkt_{partition}.parquet"
                    df_part.to_parquet(path_part, index=False)
                    mlflow.log_artifact(path_part, artifact_path="data")

                # 2) Full dataset (contient y_true !)
                path_full = "artifacts_tmp/bkt_score_full.parquet"
                cols_keep = [c for c in [
                    "ds",
                    "y_true",
                    "y_hat",
                    "lo_95",
                    "hi_95",
                    "partition",
                    "p_selected",
                ] if c in bkt_score.columns]

                bkt_score[cols_keep].to_parquet(path_full, index=False)
                mlflow.log_artifact(path_full, artifact_path="data")

            # ------------------------------------------------
            # Model logging
            # ------------------------------------------------
            model_obj = None
            model_source = None

            if "models_by_partition" in globals() and isinstance(globals()["models_by_partition"], dict):
                model_obj = globals()["models_by_partition"].get(partition, None)
                if model_obj is not None:
                    model_source = "precomputed"

            if model_obj is None:
                part_info = partition_summary_df.loc[partition_summary_df["partition"] == partition]
                if len(part_info) == 1:
                    ds_start = pd.to_datetime(part_info["ds_start"].iloc[0])
                    ds_end   = pd.to_datetime(part_info["ds_end"].iloc[0])

                    if "y" in globals():
                        model_obj = fit_final_ar_for_partition(
                            globals()["y"], ds_start, ds_end, p=12, trend=AR_PARAMS["trend"]
                        )
                        model_source = "refit_on_partition_y"
                    elif "ts_ar" in globals():
                        model_obj = fit_final_ar_for_partition(
                            globals()["ts_ar"], ds_start, ds_end, p=12, trend=AR_PARAMS["trend"]
                        )
                        model_source = "refit_on_partition_ts_ar"

            if model_obj is not None:
                model_path = f"artifacts_tmp/ar_model_{partition}.joblib"
                joblib.dump(model_obj, model_path)
                mlflow.log_artifact(model_path, artifact_path="model")
                mlflow.log_param("model_logged", True)
                mlflow.log_param("model_source", model_source)
            else:
                mlflow.log_param("model_logged", False)
                mlflow.log_param("model_source", "none_found")

print("Logging terminé")

Logging terminé


# Verification

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

EXP_NAME = "Baseline"
exp = mlflow.get_experiment_by_name(EXP_NAME)
assert exp is not None, f"Experiment introuvable: {EXP_NAME}"

runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.run_name LIKE 'AR_%'",
    output_format="pandas"
)

client = MlflowClient()

download_dir = "mlflow_downloads_baseline_ar"
os.makedirs(download_dir, exist_ok=True)

frames = []
downloaded = []

for _, r in runs.iterrows():
    run_id = str(r["run_id"])

    # Lister ce qu'il y a dans /data
    try:
        arts = client.list_artifacts(run_id, path="data")
    except Exception as e:
        print(f"[WARN] list_artifacts failed for {run_id}: {e}")
        continue

    # Télécharger les .parquet
    for a in arts:
        if a.is_dir:
            continue
        if not a.path.endswith(".parquet"):
            continue

        local_path = client.download_artifacts(run_id, a.path, dst_path=download_dir)
        downloaded.append((run_id, a.path, local_path))

        df_tmp = pd.read_parquet(local_path)
        df_tmp["run_id"] = run_id
        df_tmp["artifact_path"] = a.path
        frames.append(df_tmp)

print("Runs AR trouvés:", len(runs))
print("Parquets téléchargés:", len(downloaded))

# Dataframe global (tous bkt réunis)
bkt_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("bkt_all shape:", bkt_all.shape)

# Vérif colonnes
print("Colonnes bkt_all:", bkt_all.columns.tolist())
bkt_all.head()

Runs AR trouvés: 5
Parquets téléchargés: 9
bkt_all shape: (2568, 9)
Colonnes bkt_all: ['ds', 'y_true', 'y_hat', 'lo_95', 'hi_95', 'partition', 'p_selected', 'run_id', 'artifact_path']


,ds,y_true,y_hat,lo_95,hi_95,partition,p_selected,run_id,artifact_path
0,1990-01-01,0.0,0.038995,-0.896105,0.974096,1990-1999,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
1,1990-02-01,0.1,-0.158664,-0.963583,0.646255,1990-1999,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
2,1990-03-01,0.2,-0.338061,-1.205755,0.529634,1990-1999,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
3,1990-04-01,0.2,0.079964,-0.698902,0.858830,1990-1999,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
4,1990-05-01,0.2,-0.004353,-0.955797,0.947091,1990-1999,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet


In [ ]:
assert "bkt_all" in globals(), "bkt_all n'existe pas (exécute la cellule 1)"

df = bkt_all.copy()

# normaliser ds
if "ds" in df.columns:
    df["ds"] = pd.to_datetime(df["ds"], errors="coerce")

# y_true
if "y_true" not in df.columns and "y" in df.columns:
    df["y_true"] = df["y"]

# y_hat
# (si tu avais un autre nom, ajoute-le ici)
if "y_hat" not in df.columns and "forecast" in df.columns:
    df["y_hat"] = df["forecast"]

# intervalles 95% (si dispo)
if "lo_95" not in df.columns and "y_hat_p05" in df.columns:
    df["lo_95"] = df["y_hat_p05"]
if "hi_95" not in df.columns and "y_hat_p95" in df.columns:
    df["hi_95"] = df["y_hat_p95"]

# garder uniquement les lignes exploitables pour DM / erreurs
need = [c for c in ["ds","partition","y_true","y_hat","lo_95","hi_95","p_selected","run_id","artifact_path"] if c in df.columns]
df_use = df[need].copy()

print("Colonnes retenues:", df_use.columns.tolist())
print("NaN y_true:", df_use["y_true"].isna().sum() if "y_true" in df_use.columns else "NA")
print("NaN y_hat :", df_use["y_hat"].isna().sum() if "y_hat" in df_use.columns else "NA")

df_use.head(10)

Colonnes retenues: ['ds', 'partition', 'y_true', 'y_hat', 'lo_95', 'hi_95', 'p_selected', 'run_id', 'artifact_path']
NaN y_true: 0
NaN y_hat : 0


,ds,partition,y_true,y_hat,lo_95,hi_95,p_selected,run_id,artifact_path
0,1990-01-01,1990-1999,0.0,0.038995,-0.896105,0.974096,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
1,1990-02-01,1990-1999,0.1,-0.158664,-0.963583,0.646255,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
2,1990-03-01,1990-1999,0.2,-0.338061,-1.205755,0.529634,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
3,1990-04-01,1990-1999,0.2,0.079964,-0.698902,0.858830,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
4,1990-05-01,1990-1999,0.2,-0.004353,-0.955797,0.947091,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
5,1990-06-01,1990-1999,-0.1,0.096554,-0.861185,1.054294,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
6,1990-07-01,1990-1999,0.3,-0.003939,-0.700189,0.692311,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
7,1990-08-01,1990-1999,0.5,-0.203608,-1.088766,0.681551,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
8,1990-09-01,1990-1999,0.6,0.007901,-1.221863,1.237665,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
9,1990-10-01,1990-1999,0.6,0.064938,-1.023831,1.153706,4,4a85f69b86244df091ad640c3a96d467,data/bkt_score_full.parquet
